# 06｜MLP 决策边界闯关项目 🎮

你的任务不是把模型堆得最大，而是用有限计算预算驯服三种二维数据。最终提交一条“预测—实验—证据—结论”链。

| 关卡 | 建议时间 | 必交证据 |
|---|---:|---|
| 热身与基线 | 20 分钟 | 基线图、三星表 |
| 地图侦察 | 40 分钟 | 三种数据差异与预测 |
| 单变量实验 | 60 分钟 | 至少 3 组公平对照 |
| 阅读并修改模型 | 45 分钟 | 一处有效代码修改、烟雾测试 |
| Boss 关：预算内改进 | 45 分钟 | 最多 3 次尝试、最佳配置 |
| 证据报告 | 30 分钟 | 300–500 字结论与边界 |
| **合计** | **240 分钟** | **约 4 小时** |

四小时主要用于思考、比较、改代码和写结论；实际训练仍保持轻量。

In [ ]:
%matplotlib inline
from pathlib import Path
import sys

LAB_DIR = Path.cwd()
if not (LAB_DIR / "mlp_lab").is_dir():
    LAB_DIR = (Path.cwd() / "01_mlp_lab").resolve()
if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))

from mlp_lab import (
    base_config, challenge_report, quick_demo, show_datasets, train_experiment
)

## 闯关规则：公平比“大力出奇迹”更重要

- 固定使用 DEVICE = "cpu"；MLP 不占课堂 GPU。
- 一次只改一个变量；同组实验保持数据、seed 和训练预算一致。
- 全项目最多运行 9 次正式训练；失败的语法调试不计入。
- 不靠增加样本量或 Epoch 刷分。代码会拦截超过共享服务器上限的配置。
- 计分不使用运行时间，因为服务器繁忙程度不同会造成不公平。

In [ ]:
DEVICE = "cpu"

baseline = quick_demo(
    dataset="spiral",
    hidden_sizes=(16, 16),
    device=DEVICE,
    epochs=50,
    n_samples=450,
    show_story=False,
)
baseline_score = challenge_report(
    baseline, accuracy_target=0.90, parameter_budget=500, update_budget=400
)

### 热身检查点

在继续前写下：基线得到 __ 星；最难得到的是 __ 星；我认为原因是 __。

注意：星级只是任务约束，不等于科研结论。一次随机划分上的高准确率不能证明模型普遍更好。

## 第一关｜地图侦察：哪一种边界最难画？（40 分钟）

先观察 moons、circles、spiral。运行训练前，按“最容易 → 最困难”排序并说明理由。然后选择一种数据验证预测。

In [ ]:
show_datasets(seed=42, n_samples=360)

# 先写预测，再从 moons / circles / spiral 中选择一个。
CHOSEN_DATASET = "circles"
map_result = quick_demo(
    dataset=CHOSEN_DATASET, device=DEVICE, epochs=35, n_samples=360, show_story=False
)

### 地图记录

我的排序：__。图形证据：__。训练结果支持 / 不支持我的预测，因为 __。

不要只写 Accuracy；至少描述一处边界形状或误判点分布。

## 第二关｜单变量实验：找出真正起作用的改动（60 分钟）

下面三组实验分别只改激活函数、隐藏层宽度和优化器。运行前为每组写一句预测；运行后记录数值证据与图形证据。可以替换尝试值，但不能同时改两个字段。

In [ ]:
TRIALS = {
    "激活函数": {"activation": "tanh"},
    "隐藏层宽度": {"hidden_sizes": (8, 8)},
    "优化器": {"optimizer": "momentum"},
}

trial_results = {}
for label, one_change in TRIALS.items():
    print(f"\n===== {label}：{one_change} =====")
    trial_results[label] = quick_demo(
        dataset="spiral",
        device=DEVICE,
        epochs=50,
        n_samples=450,
        show_story=False,
        **one_change,
    )

### 公平对照记录

| 实验 | 我的预测 | 唯一修改项 | Accuracy 变化 | 参数量变化 | 图形证据 | 结论 |
|---|---|---|---:|---:|---|---|
| 激活函数 |  |  |  |  |  |  |
| 隐藏层宽度 |  |  |  |  |  |  |
| 优化器 |  |  |  |  |  |  |

如果预测错了，这不是扣分点；能够用证据解释为什么错，才是本关的目标。

## 安全门演示｜四小时工作量不等于四小时占用服务器

下面故意提交一个过大的 Epoch 配置。程序应立刻拒绝，不会开始训练。阅读错误信息，说明应该缩小哪个参数。

In [ ]:
from dataclasses import replace

unsafe = replace(base_config(), epochs=500)
try:
    train_experiment(unsafe)
except ValueError as error:
    print("安全门已拦截：", error)

## 第三关｜读代码并增加一种激活函数（45 分钟）

1. 打开 mlp_lab/model.py，先用自己的话解释 sizes、layers 和 is_output_layer。
2. 在 _activation() 的映射中增加一种 PyTorch 已提供的激活层，不新增依赖。
3. 把下面的名称改成你的新名称并运行。
4. 在终端执行 python test_mlp_smoke.py，保存通过截图。
5. 比较它与 ReLU；只陈述当前数据和配置下的现象。

这一关考查的是能否找到最小修改点，不要求重写训练循环。

In [ ]:
# 完成 model.py 修改前保持 tanh；修改后换成你新增的名称。
ACTIVATION_TO_TEST = "tanh"
extension_result = quick_demo(
    dataset="spiral",
    activation=ACTIVATION_TO_TEST,
    device=DEVICE,
    epochs=50,
    n_samples=450,
    show_story=False,
)

## Boss 关｜三次机会，预算内争取三星（45 分钟）

目标：spiral 测试 Accuracy ≥ 97%，参数量 ≤ 500，更新次数 ≤ 400。你最多尝试 3 个配置；每次尝试前必须先写预测。

可以修改 hidden_sizes、activation、optimizer、learning_rate、dropout，但保持 epochs=50、n_samples=450、seed=42。没有三星也可以完成，只要能解释瓶颈和下一步。

In [ ]:
BOSS_CONFIG = dict(
    hidden_sizes=(16, 16),
    activation="relu",
    optimizer="adam",
    learning_rate=0.02,
    dropout=0.0,
)

boss_result = quick_demo(
    dataset="spiral",
    device=DEVICE,
    epochs=50,
    n_samples=450,
    seed=42,
    show_story=False,
    **BOSS_CONFIG,
)
boss_score = challenge_report(
    boss_result, accuracy_target=0.97, parameter_budget=500, update_budget=400
)

## 最终提交｜300–500 字证据报告（30 分钟）

请按下面顺序写，不要只贴截图：

1. **问题与预测：**你原来认为哪个组件最重要，为什么？
2. **公平比较：**列出固定条件和唯一修改项。
3. **数值证据：**至少引用 Accuracy、参数量、更新次数中的两项。
4. **图形证据：**描述边界、误判点、Loss 或梯度中的一项具体变化。
5. **代码理解：**说明你修改了哪个最小位置，为什么不需要改训练循环。
6. **结论边界：**使用“在当前数据、seed 和训练预算下”限定结论。

提交清单：完成后的 Notebook、model.py 修改、烟雾测试截图、最终报告。

## 完成后：释放资源并结束内核

先保存 Notebook，再运行最后一格。继续实验时需要重新选择 Python 内核。

In [ ]:
%reset -f
import gc
import matplotlib.pyplot as plt
import torch
from IPython import get_ipython

plt.close("all")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("闯关记录已完成，正在结束当前 Notebook 内核……")
get_ipython().kernel.do_shutdown(restart=False)